In [7]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [5]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:

def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """

    messages = []
    add_user_message(messages=messages, text=prompt)
    add_assistant_message(messages=messages, text="```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [7]:
dataset = generate_dataset()
dataset

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URL. For example, 'https://my-bucket.s3.us-west-2.amazonaws.com' should return 'us-west-2'."},
 {'task': "Create a JSON object representing an IAM policy that allows a user to read and write objects in an S3 bucket named 'my-data-bucket'."},
 {'task': 'Write a regular expression that matches valid AWS Lambda function names. Function names can contain letters, numbers, hyphens, and underscores, must be between 1-64 characters, and cannot start with a hyphen.'}]

In [8]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [10]:
def run_prompt(test_case):
    """
        Merges the prompt and the test case input and then returns the result
    """

    prompt = f"""
        Please solve the following task:

        {test_case["task"]}
    """

    messages = []
    add_user_message(messages=messages, text=prompt)
    output = chat(messages=messages)
    return output

In [11]:
def run_test_case(test_case):
    """
        calls run_prompt and grades the result
    """

    output = run_prompt(test_case=test_case)

    # TODO: Need to grade
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [12]:
def run_eval(dataset):
    """
        Loads the dataset and calls run_test_case with each case
    """
    results = []
    for test_case in dataset:
        result = run_test_case(test_case=test_case)
        results.append(result)

    return results

In [13]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset=dataset)

In [15]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a comprehensive solution:\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_s3_region(s3_url: str) -> Optional[str]:\n    \"\"\"\n    Extract the AWS region from an S3 bucket URL.\n    \n    Supports multiple S3 URL formats:\n    - Virtual-hosted-style: https://my-bucket.s3.us-west-2.amazonaws.com\n    - Path-style: https://s3.us-west-2.amazonaws.com/my-bucket\n    - S3 Transfer Acceleration: https://my-bucket.s3-accelerate.amazonaws.com\n    - Legacy: https://my-bucket.s3.amazonaws.com (returns 'us-east-1')\n    \n    Args:\n        s3_url: The S3 bucket URL\n        \n    Returns:\n        The AWS region string, or None if region cannot be extracted\n    \"\"\"\n    \n    # Pattern 1: Virtual-hosted-style with region\n    # https://my-bucket.s3.us-west-2.amazonaws.com\n    match = re.search(r'\\.s3[.-]([a-z0-9\\-]+)\\.amazonaws\\.com', s3_url)\n    if match:\n        region = match.group(1)\n        